# CropHist Iowa Sample

This notebook works in two modes:

- **Default:** load the published Iowa sample directly from CropHist public URLs. No API key required.
- **Optional:** validate a small batch of those same field IDs through the live CropHist API.

That keeps the free sample usable immediately while still showing how the API fits into an analyst workflow.


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

SAMPLE_BASE = 'https://crophist.com/pages/iowa-sample/downloads'
API_BASE = os.environ.get('CROPHIST_API_BASE', 'https://api.crophist.com')
API_KEY = os.environ.get('CROPHIST_API_KEY', '').strip()

# These IDs come from the published Iowa sample export.
FIELD_IDS = [
    '5003.VJV5',
    '5003.VW5D',
    '5003.YRQW',
    '5003.Z5MQ',
]


In [ ]:
history_df = pd.read_parquet(f'{SAMPLE_BASE}/iowa-fields-sample-long.parquet')
history_df = history_df[history_df['field_id'].isin(FIELD_IDS)].copy()
history_df = history_df.sort_values(['field_id', 'year']).reset_index(drop=True)

field_count = history_df['field_id'].nunique()
print(f'Loaded {len(history_df):,} rows across {field_count} Iowa fields from the public sample.')
display(history_df.head())


In [ ]:
pivot = history_df.pivot(index='year', columns='field_id', values='crop_name')
display(pivot.tail(8))

example_field = history_df['field_id'].iloc[0]
example = history_df[history_df['field_id'] == example_field].copy()

plt.figure(figsize=(10, 4))
plt.plot(example['year'], example['crop_code'], marker='o')
plt.title(f'Crop sequence for {example_field}')
plt.xlabel('Year')
plt.ylabel('Crop code')
plt.grid(alpha=0.2)
plt.show()


In [ ]:
rotation_summary = (
    history_df
    .assign(is_corn_soy=lambda df: df['crop_name'].isin(['Corn', 'Soybeans']))
    .groupby('field_id')
    .agg(
        years=('year', 'count'),
        corn_soy_share=('is_corn_soy', 'mean'),
        last_crop=('crop_name', 'last'),
    )
    .reset_index()
)

rotation_summary.sort_values('corn_soy_share', ascending=False).head()


In [ ]:
def baseline_forecast(last_crop):
    if last_crop == 'Corn':
        return 'Soybeans'
    if last_crop == 'Soybeans':
        return 'Corn'
    return last_crop


rotation_summary['forecast_2026'] = rotation_summary['last_crop'].map(baseline_forecast)
forecast_mix = rotation_summary['forecast_2026'].value_counts(normalize=True).rename('share')
forecast_mix.head(10)


## Optional: Validate The Same IDs Through The Live API

Run the next cell only if you have a valid `CROPHIST_API_KEY`. If not, skip it. The analysis above already works entirely from the public sample files.


In [ ]:
if not API_KEY:
    print('Skipping live API validation. Set CROPHIST_API_KEY in Colab if you want to test POST /v1/fields.')
else:
    response = requests.post(
        f'{API_BASE}/v1/fields',
        headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'},
        json={'field_ids': FIELD_IDS},
        timeout=30,
    )
    if response.status_code == 403:
        print('CropHist API returned 403 Forbidden. Check CROPHIST_API_KEY, then rerun this optional validation cell.')
    else:
        response.raise_for_status()
        api_results = response.json()['results']
        print(f'Validated {len(api_results)} fields through the live API.')
        display(pd.DataFrame(api_results).head())
